# Reto 02 RA5 - Exploracion del Pipeline Medallon IoT

**Alumno:** autor  
**Modulo:** Big Data Aplicado (UT5)

Este notebook recorre **MANUALMENTE** las tres capas del medallon para validar que cada fase del pipeline ha funcionado.  
Se ejecuta despues del DAG `iot_medallion_pipeline` desde Airflow.

**Capas a inspeccionar:**
1. Bronze - JSONL crudo en HDFS (con errores intencionados)
2. Silver - Parquet limpio en MinIO (sin errores, tipos correctos)
3. Gold - Parquet agregado en MinIO (KPIs para Superset)
4. Calidad - Informe JSON + cuarentena en MinIO

**Patron de teoria reusado:**
- HDFS: `hdfs.InsecureClient` (notebook UT5/01_Demo_HDFS_S3 del profesor)
- MinIO: `pandas.read_parquet` con `storage_options` (mismo notebook)
- Comparativa entre capas demuestra los principios de la arquitectura medallon (UT5)

In [ ]:
# Imports comunes y constantes del proyecto
import sys
sys.path.insert(0, '/home/jovyan/work/src/jobs')

import json
import pandas as pd
import matplotlib.pyplot as plt
from hdfs import InsecureClient
import s3fs

from lib_paths import (
    BRONZE_HDFS_BASE, DEVICE_ID,
    HDFS_NAMENODE_HTTP,
    SILVER_S3_BASE, GOLD_S3_BASE,
    QUALITY_S3_BASE, QUARANTINE_S3_BASE,
    PANDAS_S3_STORAGE_OPTIONS, MINIO_BUCKET,
    MINIO_ACCESS_KEY, MINIO_SECRET_KEY, MINIO_ENDPOINT,
)

FECHA = '2026-04-27'   # cambiar segun la fecha del run del DAG
print(f'Sensor: {DEVICE_ID}, fecha analizada: {FECHA}')

## 1. Capa Bronze (HDFS) - dato CRUDO con errores

Comprobamos:
- Que existen los ficheros en la particion `year=YYYY/month=MM/day=DD/`
- Que los registros INCLUYEN errores (status=ERR), porque Bronze NO transforma

Justificacion de no limpiar aqui (UT5): si limpiaramos en la ingesta, perderiamos trazabilidad de QUE llego mal. La capa de calidad lo hara explicito en el informe.

In [ ]:
# Listamos la particion del dia
client = InsecureClient(HDFS_NAMENODE_HTTP, user='jovyan')
y, m, d = FECHA.split('-')
ruta = f'{BRONZE_HDFS_BASE}/year={y}/month={m}/day={d}'
print('Contenido de', ruta)
for f in client.list(ruta, status=False):
    print('   -', f)

In [ ]:
# Leemos el primer fichero JSONL como pandas DataFrame
ficheros = client.list(ruta, status=False)
primero = f'{ruta}/{ficheros[0]}'
with client.read(primero, encoding='utf-8') as r:
    bronze_df = pd.read_json(r, lines=True)

print(f'Filas en Bronze: {len(bronze_df)}')
print('Distribucion del campo status (debe haber OK + ERR):')
print(bronze_df['status'].value_counts())
bronze_df.head()

## 2. Capa de calidad - informe JSON + cuarentena

El validador escribio dos cosas en MinIO:
- `quality/iot/.../report.json` - resumen agregado
- `quarantine/iot/.../quarantine.jsonl` - los registros que fallaron, con `_quality_errors`

Este informe va incluido tal cual en el PDF de la memoria como evidencia del Ejercicio 3 (2 pt).

In [ ]:
fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    client_kwargs={'endpoint_url': MINIO_ENDPOINT},
)
informe_path = f'{MINIO_BUCKET}/quality/iot/{DEVICE_ID}/{FECHA}/report.json'
with fs.open(informe_path, 'r', encoding='utf-8') as f:
    informe = json.load(f)
print(json.dumps(informe, indent=2, ensure_ascii=False))

In [ ]:
# Cuarentena: primeras 10 filas de los registros invalidos
cuar_path = f'{MINIO_BUCKET}/quarantine/iot/{DEVICE_ID}/{FECHA}/quarantine.jsonl'
if fs.exists(cuar_path):
    with fs.open(cuar_path, 'r', encoding='utf-8') as f:
        invalidos = [json.loads(l) for l in f]
    print(f'{len(invalidos)} registros en cuarentena. Ejemplos:')
    pd.DataFrame(invalidos).head(10)
else:
    print('No hay cuarentena (todo paso las reglas)')

## 3. Capa Silver (MinIO Parquet) - dato LIMPIO y tipado

Aqui ya:
- No hay status=ERR (filtrados)
- timestamp es datetime64
- temperatura/humedad/co2 estan en rango fisico
- No hay duplicados de event_id (UT3.4 idempotencia)

In [ ]:
# Leemos Silver con pandas + storage_options (patron del profesor, notebook 01)
silver = pd.read_parquet(SILVER_S3_BASE, storage_options=PANDAS_S3_STORAGE_OPTIONS)
print('Filas en Silver:', len(silver))
print('\nTipos:')
print(silver.dtypes)
silver.head()

In [ ]:
# Verificacion rapida: rangos fisicos correctos + sin duplicados
print('Rangos:')
print(silver[['temperature','humidity','co2','battery']].describe().loc[['min','max']])
print(f'\nUnicidad event_id: {silver.event_id.nunique()} unicos / {len(silver)} filas')

## 4. Capa Gold (MinIO Parquet) - 3 datasets de KPIs

Listos para Superset/DuckDB.

In [ ]:
for ds in ['hourly_metrics', 'daily_summary', 'anomalies']:
    df = pd.read_parquet(f'{GOLD_S3_BASE}/{ds}', storage_options=PANDAS_S3_STORAGE_OPTIONS)
    print(f'== {ds:18s} {len(df):4d} filas ==')
    display(df.head())
    print()

## 5. Visualizacion rapida (preview de lo que vera Superset)

Una grafica con matplotlib para confirmar que los datos tienen sentido analitico.

In [ ]:
hourly = pd.read_parquet(f'{GOLD_S3_BASE}/hourly_metrics', storage_options=PANDAS_S3_STORAGE_OPTIONS)
hourly = hourly.sort_values(['year','month','day','hour'])

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(hourly['hour'], hourly['temp_avg'], marker='o', label='temp_avg')
axes[0].fill_between(hourly['hour'], hourly['temp_min'], hourly['temp_max'], alpha=0.2)
axes[0].set_ylabel('Temperatura (C)')
axes[0].axhspan(18, 28, alpha=0.1, color='green', label='zona confort')
axes[0].legend()

axes[1].plot(hourly['hour'], hourly['hum_avg'], marker='s', color='steelblue')
axes[1].set_ylabel('Humedad (%)')

axes[2].plot(hourly['hour'], hourly['co2_avg'], marker='^', color='darkred')
axes[2].axhline(1000, color='red', linestyle='--', label='umbral CO2')
axes[2].set_ylabel('CO2 (ppm)')
axes[2].set_xlabel('Hora del dia')
axes[2].legend()

plt.suptitle(f'Sensor {DEVICE_ID} - {FECHA}')
plt.tight_layout()
plt.show()

## 6. Conclusion del recorrido

Si TODAS las celdas anteriores se ejecutaron sin error:
- El pipeline funciona end-to-end
- Los datos crudos siguen accesibles en HDFS (Bronze)
- La capa de calidad ha aislado los registros invalidos en cuarentena
- Silver tiene datos limpios y tipados
- Gold tiene los 3 datasets agregados listos para BI

**Siguiente paso**: cambiar perfiles Docker para arrancar Superset y conectar DuckDB a MinIO. Ver `src/docs/manual_ejecucion.md`.